# 03 · Hashing and sliding window

After this notebook you can trade memory for time with a hash map (`dict`, `set`, `Counter`, `defaultdict`), solve "best contiguous stretch" problems in O(n) with fixed and variable sliding windows, and recognise when a window does *not* work and prefix sums + a hash map do.

**Time:** ~40 min · **Runs:** offline on CPU in < 1 min · **Needs:** [02 · Arrays and two pointers](02_arrays_and_two_pointers.ipynb) · **Theory:** [DSA basics course](../../dsa_basics/index.html), chapter 6 "Hash tables"

## Why this matters in interviews

- A hash map is the single most common way to turn O(n²) into O(n): "have I seen this before?" becomes an O(1) lookup instead of a scan.
- Sliding-window questions ("longest substring without...", "minimum window containing...") are among the most-asked medium/hard string problems — one template makes them mechanical.
- The senior follow-up is knowing when a window *fails* (negative numbers) and switching to prefix sums + a hash map.

## What interviewers ask

- "Two Sum on an *unsorted* array, in one pass." (LeetCode 1)
- "Is there a duplicate within distance k?" (LC 219)
- "Maximum sum of any k consecutive elements." (fixed window, LC 643)
- "Longest substring without repeating characters." (LC 3)
- "Longest run of one character if you may replace up to k characters." (LC 424)
- "Minimum window of s that contains every character of t." (LC 76)
- "Count subarrays that sum to k — the array has negative numbers." (LC 560)
- "Group anagrams." (LC 49)
- Follow-ups: "What is the worst case of a hash lookup?" · "Why does the window break with negative numbers?" · "What would you use as the key?"

In [ ]:
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

random.seed(0)
rng = random.Random(1)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=COLORS), "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2})


def interleaved_times(calls, number=1, rounds=2):
    """Seconds per call for each fn in `calls`: best over rounds, alternating calls so background load hits all equally."""
    best = [float("inf")] * len(calls)
    for _ in range(rounds):
        for i, call in enumerate(calls):
            t0 = time.perf_counter()
            for _ in range(number):
                call()
            best[i] = min(best[i], (time.perf_counter() - t0) / number)
    return best

## Pattern 1: hashing — remember everything you have seen

**In plain English:** a hash map answers "what do I know about key `k`?" in O(1) on average, so one pass can remember every value, index or count seen so far — instead of re-scanning the array for each element.

How it works: `hash(key)` turns the key into an integer, which picks a slot in an array; collisions are resolved by probing other slots, and the table grows to stay mostly empty. That is why lookups are O(1) *on average* but O(n) in the worst case (every key colliding).

```
   key "eat" ──hash()──> 8246...  ──mod table size──> slot 5 ──> ("eat", ["eat", "tea", "ate"])
```

**Recognise it when you see:** pair / complement, "seen before", duplicates, counts or frequencies, "group by", anagrams, "first unique", last index of something.

**Python toolbox:** `dict` (with `.get(k, default)`), `set`, `collections.Counter` (counts, `most_common`), `collections.defaultdict(list)` (grouping).

**Template:**

```python
seen = {}                          # value -> index (or count, or list)
for i, x in enumerate(items):
    if complement(x) in seen:      # O(1) average: ask before you insert
        return seen[complement(x)], i
    seen[x] = i
```

Two rules that trip people up — keys must be **hashable** (immutable), and **set order of strings is not stable between runs** (Python randomises string hashes per process):

In [ ]:
counts_by_letters = {("a", "e", "t"): 3}          # a tuple is hashable: fine as a key
print("tuple as a key ->", counts_by_letters)
try:
    bad = {["a", "e", "t"]: 3}                     # a list is mutable: not hashable
except TypeError as err:
    print("list as a key ->", err)

words = ["eat", "tea", "tan"]
print("dict keeps insertion order:", list(dict.fromkeys(words)))
print("sorted(set(...)), same every run:", sorted(set(words)), "  <- raw set order of strings can change between runs")
print("hash(42) =", hash(42), " (small ints hash to themselves; str hashes are salted per process)")

## Pattern 2: sliding window — one pass over contiguous ranges

**In plain English:** for "best contiguous stretch" problems, keep a window `[left, right]` and slide it along. Add the element that enters on the right, remove the one that leaves on the left, and update the answer. Every element enters once and leaves once, so the scan is O(n) — instead of recomputing each of the O(n²) windows from scratch.

- **Fixed size k:** both ends move together.
- **Variable size:** grow on the right; while the window breaks the rule, shrink from the left; then record the answer.

The variable window needs one property: **validity must be monotone** — if a window is valid, every smaller window inside it is valid too (for "longest"), or growing it never makes it invalid again (for "shortest"). That is what lets the left pointer only ever move right.

**Templates:**

```python
def fixed_window(a, k):
    window = sum(a[:k]); best = window
    for right in range(k, len(a)):
        window += a[right] - a[right - k]          # one enters, one leaves
        best = max(best, window)
    return best

def variable_window(s):
    counts, left, best = {}, 0, 0
    for right, x in enumerate(s):
        counts[x] = counts.get(x, 0) + 1           # 1. expand: take s[right]
        while window_is_invalid(counts):           # 2. shrink until valid again
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)         # 3. record the answer for this right end
    return best
```

A helper to draw a window moving over an array — one row per step, the window in blue, the element that just entered in dark blue:

In [ ]:
def draw_window_trace(items, steps, title):
    """steps: list of (left, right, note); the window is items[left..right] and items[right] just entered."""
    fig, ax = plt.subplots(figsize=(10, min(4.5, 0.5 * len(steps) + 0.9)))
    for row, (left, right, note) in enumerate(steps):
        y = len(steps) - 1 - row
        for i, v in enumerate(items):
            if left <= i <= right:
                face, ink = (COLORS[0], "white") if i == right else ("#b7d3f6", "#0b0b0b")
            else:
                face, ink = "#f0efec", "#898781"
            ax.add_patch(plt.Rectangle((i, y), 0.92, 0.8, facecolor=face, edgecolor="none"))
            ax.text(i + 0.46, y + 0.4, str(v), ha="center", va="center", color=ink, fontsize=10)
        ax.text(len(items) + 0.3, y + 0.4, note, va="center", fontsize=9)
    ax.set_xlim(-0.2, len(items) + 7)
    ax.set_ylim(-0.2, len(steps))
    ax.axis("off")
    ax.set_title(title)
    plt.show()

## Problem 1 · Two Sum with a hash map (LC 1)

**Statement:** return the indices of the two numbers in the *unsorted* `nums` that add up to `target`. `[2, 7, 11, 15]`, 9 → `[0, 1]`; `[3, 2, 4]`, 6 → `[1, 2]`.

**Brute force:** every pair, O(n²). (Sorting + two pointers is O(n log n) but loses the original indices.) **Optimal:** one pass; for each `x`, ask whether its complement `target − x` has been seen, *then* store `x`.

In [ ]:
def two_sum_brute(nums, target):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
    return []


def two_sum(nums, target):
    index_of = {}                              # value -> index where we saw it
    for i, x in enumerate(nums):
        if target - x in index_of:             # the complement appeared earlier
            return [index_of[target - x], i]
        index_of[x] = i                        # insert AFTER checking, so x never pairs with itself
    return []


for (nums, target), expected in [(([2, 7, 11, 15], 9), [0, 1]), (([3, 2, 4], 6), [1, 2]), (([3, 3], 6), [0, 1]),
                                 (([1, 2], 4), []), (([], 1), []), (([-3, 4, 3, 90], 0), [0, 2])]:
    assert two_sum(nums, target) == expected, (nums, target)
for _ in range(500):
    nums = [rng.randint(-15, 15) for _ in range(rng.randint(0, 12))]
    target = rng.randint(-30, 30)
    got = two_sum(nums, target)
    assert (got == []) == (two_sum_brute(nums, target) == [])
    if got:
        assert got[0] < got[1] and nums[got[0]] + nums[got[1]] == target
print("two_sum: hand-written cases (incl. [3, 3] -> 6) + 500 random inputs agree with the brute force")

**Complexity:** O(n) time on average, O(n) extra space. **Trap:** inserting `x` *before* the check makes `[3]` with target 6 "find" the pair (0, 0).

## Problem 2 · Duplicate within distance k (LC 219)

**Statement:** is there `i ≠ j` with `nums[i] == nums[j]` and `|i − j| ≤ k`? `[1,2,3,1]`, k=3 → True; `[1,2,3,1,2,3]`, k=2 → False.

**Brute force:** compare each element with the next k — O(n·k). **Optimal A:** remember the last index of every value — O(n) time, O(n) space. **Optimal B:** a sliding window: keep a set of the last k values — O(n) time, O(min(n, k)) space.

In [ ]:
def nearby_duplicate_brute(nums, k):
    return any(nums[i] == nums[j] for i in range(len(nums)) for j in range(i + 1, min(len(nums), i + k + 1)))


def nearby_duplicate(nums, k):
    last_index = {}
    for i, x in enumerate(nums):
        if x in last_index and i - last_index[x] <= k:
            return True
        last_index[x] = i
    return False


def nearby_duplicate_window(nums, k):
    window = set()                              # the values of the last k positions
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if len(window) > k:
            window.remove(nums[i - k])          # slide: the element k steps back leaves
    return False


for (nums, k), expected in [(([1, 2, 3, 1], 3), True), (([1, 0, 1, 1], 1), True), (([1, 2, 3, 1, 2, 3], 2), False),
                            (([], 0), False), (([1, 1], 0), False), (([99, 99], 2), True)]:
    assert nearby_duplicate(nums, k) == nearby_duplicate_window(nums, k) == expected, (nums, k)
for _ in range(500):
    nums, k = [rng.randint(0, 8) for _ in range(rng.randint(0, 15))], rng.randint(0, 6)
    assert nearby_duplicate(nums, k) == nearby_duplicate_window(nums, k) == nearby_duplicate_brute(nums, k)
print("nearby_duplicate (dict) and nearby_duplicate_window (set of last k) agree with the brute force")

## Problem 3 · Maximum sum of k consecutive elements (fixed window, LC 643)

**Statement:** the largest sum of any `k` consecutive elements (1 ≤ k ≤ n). `[2, 1, 5, 1, 3, 2]`, k=3 → 9 (`5 + 1 + 3`).

**Brute force:** sum every window — O(n·k). **Optimal:** keep the window sum and update it by *one addition and one subtraction* per step — O(n), independent of k.

In [ ]:
def max_sum_k_brute(a, k):
    return max(sum(a[i:i + k]) for i in range(len(a) - k + 1))


def max_sum_k(a, k, trace=None):
    window = sum(a[:k])
    best = window
    if trace is not None:
        trace.append((0, k - 1, f"first window: sum = {window}"))
    for right in range(k, len(a)):
        window += a[right] - a[right - k]              # a[right] enters, a[right - k] leaves
        best = max(best, window)
        if trace is not None:
            trace.append((right - k + 1, right, f"+{a[right]} -{a[right - k]}  ->  sum = {window}, best = {best}"))
    return best


for (a, k), expected in [(([2, 1, 5, 1, 3, 2], 3), 9), (([2, 3, 4, 1, 5], 2), 7), (([5], 1), 5),
                         (([-1, -2, -3], 2), -3), (([4, 4, 4], 3), 12)]:
    assert max_sum_k(a, k) == expected == max_sum_k_brute(a, k), (a, k)
for _ in range(300):
    a = [rng.randint(-9, 9) for _ in range(rng.randint(1, 20))]
    k = rng.randint(1, len(a))
    assert max_sum_k(a, k) == max_sum_k_brute(a, k)
print("max_sum_k: edge cases (k = 1, k = n, all negative) + 300 random inputs pass")

steps = []
max_sum_k([2, 1, 5, 1, 3, 2], 3, steps)
draw_window_trace([2, 1, 5, 1, 3, 2], steps, "Fixed window, k = 3: one element enters, one leaves, the sum updates in O(1)")

The point of the window is that its cost does not depend on k. Measure both versions on 10,000 numbers as the window grows:

In [ ]:
a = [rng.randint(-100, 100) for _ in range(10_000)]
window_sizes = [10, 30, 100, 300, 1_000]
t_brute = interleaved_times([lambda k=k: max_sum_k_brute(a, k) for k in window_sizes])
t_window = interleaved_times([lambda k=k: max_sum_k(a, k) for k in window_sizes])
slope_brute = np.polyfit(np.log10(window_sizes), np.log10(t_brute), 1)[0]
slope_window = np.polyfit(np.log10(window_sizes), np.log10(t_window), 1)[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.loglog(window_sizes, np.array(t_brute) * 1e3, marker="o", label=f"re-sum every window, O(n·k) (slope {slope_brute:.1f})")
ax.loglog(window_sizes, np.array(t_window) * 1e3, marker="o", label=f"sliding window, O(n) (slope {slope_window:.1f})")
ax.set(title="n = 10,000: the sliding window's cost does not grow with k", xlabel="window size k (log scale)",
       ylabel="time, ms (log scale)")
ax.minorticks_off()
ax.set_xticks(window_sizes, [f"{k:,}" for k in window_sizes])
ax.legend()
plt.show()
print(f"at k = 1,000 the sliding window is {t_brute[-1] / t_window[-1]:.0f}x faster")
assert slope_brute > 0.5 and slope_window < 0.3 and t_window[-1] < t_brute[-1]

At small k the two are close — the brute force's inner `sum` runs in C — but its cost grows linearly with k while the window's stays flat.

## Problem 4 · Longest substring without repeating characters (LC 3)

**Statement:** the length of the longest substring with all-distinct characters. `"abcabcbb"` → 3 (`"abc"`), `"bbbbb"` → 1, `"pwwkew"` → 3 (`"wke"`).

**Brute force:** from every start, extend until a repeat — O(n²) (O(n³) if you re-check each substring). **Optimal:** variable window + a dict of the last index of each character. When `s[right]` already occurs *inside* the window, jump `left` just past that earlier copy.

In [ ]:
def longest_unique_brute(s):
    best = 0
    for i in range(len(s)):
        seen = set()
        for j in range(i, len(s)):
            if s[j] in seen:
                break
            seen.add(s[j])
        best = max(best, len(seen))
    return best


def longest_unique(s, trace=None):
    last_seen, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= left:    # the repeat is inside the window
            left = last_seen[ch] + 1                      # jump past the earlier copy
        last_seen[ch] = right
        best = max(best, right - left + 1)
        if trace is not None:
            trace.append((left, right, f"window '{s[left:right + 1]}', best = {best}"))
    return best


for s, expected in [("abcabcbb", 3), ("bbbbb", 1), ("pwwkew", 3), ("", 0), (" ", 1), ("abba", 2), ("dvdf", 3), ("tmmzuxt", 5)]:
    assert longest_unique(s) == expected == longest_unique_brute(s), s
for _ in range(500):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(0, 14)))
    assert longest_unique(s) == longest_unique_brute(s)
print("longest_unique: edge cases (incl. the 'abba' trap) + 500 random strings agree with the brute force")

steps = []
longest_unique("pwwkew", steps)
draw_window_trace(list("pwwkew"), steps, "Variable window on 'pwwkew': the left edge jumps past a repeated character")

**Complexity:** O(n) time; O(min(n, alphabet)) space. **The `abba` trap:** at the final `a`, its last index (0) is *left of* the window (`left` is 2). Without the `last_seen[ch] >= left` check, `left` would jump *backwards* to 1 and report 3.

## Problem 5 · Longest repeating character replacement (LC 424)

**Statement:** you may replace up to `k` characters of `s`. Return the length of the longest substring that can be made of one repeated letter. `"AABABBA"`, k=1 → 4 (`"AABA"` → `"AAAA"`).

**Key insight:** a window can be made uniform iff `window length − count of its most frequent letter ≤ k` — you replace everything that is not the majority letter. That rule is monotone (shrinking a valid window keeps it valid), so a variable window works.

In [ ]:
def replacement_brute(s, k):
    best = 0
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            if (j - i) - max(Counter(s[i:j]).values()) <= k:
                best = max(best, j - i)
    return best


def character_replacement(s, k):
    """Clear version: O(26·n) — recompute the top count each step."""
    counts, left, best = Counter(), 0, 0
    for right, ch in enumerate(s):
        counts[ch] += 1
        while (right - left + 1) - max(counts.values()) > k:    # more than k letters would need replacing
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


def character_replacement_fast(s, k):
    """Interview trick: keep max_freq as a running maximum and never decrease it — O(n).
    A stale max_freq can only keep the window from shrinking; the answer grows only when max_freq truly grows."""
    counts, left, max_freq, best = Counter(), 0, 0, 0
    for right, ch in enumerate(s):
        counts[ch] += 1
        max_freq = max(max_freq, counts[ch])
        if (right - left + 1) - max_freq > k:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


for (s, k), expected in [(("ABAB", 2), 4), (("AABABBA", 1), 4), (("", 0), 0), (("AAAA", 0), 4), (("ABCDE", 1), 2), (("ABBB", 0), 3)]:
    assert character_replacement(s, k) == character_replacement_fast(s, k) == expected, (s, k)
for _ in range(400):
    s, k = "".join(rng.choice("ABC") for _ in range(rng.randint(0, 12))), rng.randint(0, 3)
    assert character_replacement(s, k) == character_replacement_fast(s, k) == replacement_brute(s, k)
print("character_replacement (clear) and _fast (running max) both agree with the brute force")

## Problem 6 · Minimum window substring (LC 76)

**Statement:** the shortest substring of `s` containing every character of `t`, counting duplicates. `s = "ADOBECODEBANC"`, `t = "ABC"` → `"BANC"`. Return `""` if none.

**Brute force:** check every substring's letter counts — O(n²) substrings × O(n) each. **Optimal:** variable window, but for a *shortest* answer: grow right until the window covers `t`, then shrink left *while it still covers*, recording each valid window. Track coverage in O(1) with `need` (a Counter of what `t` still needs; negative means surplus) and `missing` (how many characters of `t` are not yet covered).

In [ ]:
def min_window_brute(s, t):
    need, best = Counter(t), ""
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            covers = not (need - Counter(s[i:j]))          # Counter subtraction keeps only positive counts
            if covers and (not best or j - i < len(best)):
                best = s[i:j]
    return best


def min_window(s, t):
    if not s or not t:
        return ""
    need, missing = Counter(t), len(t)
    left, best = 0, (float("inf"), 0, 0)
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1                     # this character covers something t needs
        need[ch] -= 1
        while missing == 0:                  # the window covers t: record it, then try to shrink
            if right - left + 1 < best[0]:
                best = (right - left + 1, left, right + 1)
            need[s[left]] += 1
            if need[s[left]] > 0:
                missing += 1                 # we just dropped a needed character
            left += 1
    return s[best[1]:best[2]] if best[0] != float("inf") else ""


print(repr(min_window("ADOBECODEBANC", "ABC")))
for (s, t), expected in [(("ADOBECODEBANC", "ABC"), "BANC"), (("a", "a"), "a"), (("a", "aa"), ""), (("aa", "aa"), "aa"),
                         (("ab", "b"), "b"), (("", "a"), ""), (("bba", "ab"), "ba")]:
    assert min_window(s, t) == expected, (s, t)
for _ in range(400):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(0, 10)))
    t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
    got, ref = min_window(s, t), min_window_brute(s, t)
    assert len(got) == len(ref) and (got == "" or not (Counter(t) - Counter(got)))   # same length; a non-empty answer covers t
print("min_window: edge cases + 400 random pairs match the brute force's length and cover t")

**Complexity:** O(|s| + |t|) time — each index enters and leaves the window once; O(alphabet) space.

## Problem 7 · Subarray sum equals k (LC 560) — when windows fail

**Statement:** count the contiguous subarrays whose sum is `k`. The array may contain **negative numbers** and zeros. `[1, 1, 1]`, k=2 → 2; `[1, 2, 3]`, k=3 → 2 (`[1,2]` and `[3]`).

**Why not a sliding window?** "Shrink while the sum is too big" assumes adding an element never decreases the sum. With negatives that is false, so the left pointer can skip valid answers. Show, then explain:

In [ ]:
def subarray_sum_window(nums, k):
    """Sliding window — correct ONLY for positive numbers."""
    count = left = total = 0
    for right, x in enumerate(nums):
        total += x
        while total > k and left <= right:
            total -= nums[left]
            left += 1
        count += total == k
    return count


def subarray_sum_brute(nums, k):
    count = 0
    for i in range(len(nums)):
        total = 0
        for j in range(i, len(nums)):
            total += nums[j]
            count += total == k
    return count


print("window on [4, -2, 3], k=3:", subarray_sum_window([4, -2, 3], 3), "  correct answer:", subarray_sum_brute([4, -2, 3], 3))
assert subarray_sum_window([4, -2, 3], 3) != subarray_sum_brute([4, -2, 3], 3)
for _ in range(300):                                   # ...but with positive numbers the window is right
    nums = [rng.randint(1, 5) for _ in range(rng.randint(0, 12))]
    k = rng.randint(1, 12)
    assert subarray_sum_window(nums, k) == subarray_sum_brute(nums, k)
print("with positive numbers only, the window agrees with the brute force on 300 random inputs")

At 4 the sum is too big, so the window drops the 4 — and never sees the lone `[3]` later, because the −2 kept the running sum small. The fix combines [prefix sums](02_arrays_and_two_pointers.ipynb) with a hash map: a subarray `nums[i..j]` sums to `k` exactly when `prefix[j+1] − prefix[i] = k`, i.e. when an *earlier* prefix sum equals `current − k`. Count earlier prefix sums in a dict as you go:

In [ ]:
def subarray_sum(nums, k):
    count, prefix = 0, 0
    seen = {0: 1}                          # prefix sum -> times seen; the empty prefix counts once
    for x in nums:
        prefix += x
        count += seen.get(prefix - k, 0)   # each earlier prefix equal to prefix - k ends a valid subarray here
        seen[prefix] = seen.get(prefix, 0) + 1
    return count


for (nums, k), expected in [(([1, 1, 1], 2), 2), (([1, 2, 3], 3), 2), (([1, -1, 0], 0), 3), (([], 0), 0),
                            (([3], 3), 1), (([0, 0, 0], 0), 6), (([4, -2, 3], 3), 1)]:
    assert subarray_sum(nums, k) == expected == subarray_sum_brute(nums, k), (nums, k)
for _ in range(500):
    nums = [rng.randint(-4, 4) for _ in range(rng.randint(0, 14))]
    k = rng.randint(-5, 5)
    assert subarray_sum(nums, k) == subarray_sum_brute(nums, k)
print("subarray_sum (prefix sum + hash map): edge cases + 500 random inputs with negatives agree with the brute force")

**Complexity:** O(n) time, O(n) space. **Trap:** forgetting `seen = {0: 1}` misses every subarray that starts at index 0. The same trick answers "longest subarray with sum k" (store the *first* index of each prefix sum) and "subarrays divisible by k" (key on `prefix % k`).

## Problem 8 · Group anagrams (LC 49)

**Statement:** group words that are anagrams of each other. `["eat","tea","tan","ate","nat","bat"]` → `[["eat","tea","ate"], ["tan","nat"], ["bat"]]` (any order).

**Brute force:** compare every word against every group — O(n²·L log L). **Optimal:** give every word a **canonical key** that anagrams share, and group with `defaultdict(list)`. Key options: the sorted letters (O(L log L) per word), or a tuple of 26 letter counts (O(L)). The key must be hashable, which is why it is a `str` or a `tuple`, never a `list`.

In [ ]:
def group_anagrams(words):
    groups = defaultdict(list)
    for w in words:
        groups["".join(sorted(w))].append(w)          # 'eat', 'tea', 'ate' all -> 'aet'
    return list(groups.values())


def group_anagrams_counts(words):
    groups = defaultdict(list)
    for w in words:
        key = [0] * 26
        for ch in w:
            key[ord(ch) - ord("a")] += 1
        groups[tuple(key)].append(w)                  # tuple: hashable; O(L) per word, no sort
    return list(groups.values())


def canonical(groups):                                # order-insensitive form for comparing answers
    return sorted(sorted(g) for g in groups)


print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
for words, expected in [(["eat", "tea", "tan", "ate", "nat", "bat"], [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]),
                        ([""], [[""]]), (["a"], [["a"]]), (["ab", "ba", "abc", "cab", "b"], [["ab", "ba"], ["abc", "cab"], ["b"]])]:
    assert canonical(group_anagrams(words)) == canonical(group_anagrams_counts(words)) == expected
print("group_anagrams: sorted-letters key and letter-count key give the same groups")

**Complexity:** O(n · L log L) with the sorted key, O(n · L) with the count key; O(n · L) space. In a GenAI system the same idea — a canonical key into a hash map — is an **exact-match cache**: normalise the prompt (strip, lowercase, sort parameters), hash it, look it up ([16_production/03 · Caching](../16_production/03_caching_exact_and_semantic.ipynb)).

## Pattern cheat sheet — if you see X, think Y

| If you see... | Think... |
|---|---|
| pair / complement in an **unsorted** array | dict value → index; check before you insert |
| "seen before", duplicates, first repeat | `set` |
| counts, frequencies, majority, anagram check | `Counter` (or a 26-slot list for lowercase letters) |
| group things that share a property | `defaultdict(list)` keyed by a canonical form (sorted string, tuple of counts) |
| "within distance k" | last-index dict, or a set of the last k items |
| every window of size k | fixed window: add the right element, subtract the left |
| "longest / shortest substring or subarray such that..." | variable window: expand right, shrink left while invalid (or while valid, for shortest) |
| a window must *cover* a multiset | `need` Counter + a `missing` count |
| count subarrays with sum k, negatives allowed | prefix sum + dict of prefix-sum counts, `seen = {0: 1}` |
| window rule is not monotone (negatives) | not a window problem — prefix sums + hash map |

## Try it yourself

**1. First unique character (LC 387).** Return the index of the first character that appears exactly once in `s`, or −1: `"leetcode"` → 0, `"loveleetcode"` → 2, `"aabb"` → −1.

In [ ]:
# Solution — try it yourself first, then run this
def first_unique_char(s):
    counts = Counter(s)                      # pass 1: count
    for i, ch in enumerate(s):               # pass 2: first with count 1
        if counts[ch] == 1:
            return i
    return -1


for s, expected in [("leetcode", 0), ("loveleetcode", 2), ("aabb", -1), ("", -1), ("z", 0), ("aab", 2)]:
    assert first_unique_char(s) == expected, s
print("first_unique_char: all cases pass (O(n) time, O(alphabet) space)")

**2. Permutation in string (LC 567).** Does `s2` contain a permutation of `s1` as a substring? `s1 = "ab"`, `s2 = "eidbaooo"` → True (`"ba"`). Use a fixed window of length `len(s1)` and compare letter counts.

In [ ]:
# Solution — try it yourself first, then run this
def check_inclusion(s1, s2):
    k = len(s1)
    if k > len(s2):
        return False
    need, window = Counter(s1), Counter(s2[:k])
    if window == need:
        return True
    for right in range(k, len(s2)):
        window[s2[right]] += 1                   # enters
        window[s2[right - k]] -= 1               # leaves (a count of 0 still compares equal to "missing", 3.10+)
        if window == need:                       # O(alphabet) per step
            return True
    return False


def check_inclusion_brute(s1, s2):
    return any(sorted(s2[i:i + len(s1)]) == sorted(s1) for i in range(len(s2) - len(s1) + 1))


for (s1, s2), expected in [(("ab", "eidbaooo"), True), (("ab", "eidboaoo"), False), (("a", "a"), True), (("abc", "ab"), False), (("adc", "dcda"), True)]:
    assert check_inclusion(s1, s2) == expected, (s1, s2)
for _ in range(300):
    s1 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 3)))
    s2 = "".join(rng.choice("abc") for _ in range(rng.randint(0, 10)))
    assert check_inclusion(s1, s2) == check_inclusion_brute(s1, s2)
print("check_inclusion: all cases + 300 random pairs pass")

**3. Minimum size subarray sum (LC 209).** All numbers are positive. Return the length of the *shortest* contiguous subarray with sum ≥ `target`, or 0: `target = 7`, `[2,3,1,2,4,3]` → 2 (`[4,3]`). (Positive numbers → the window rule is monotone.)

In [ ]:
# Solution — try it yourself first, then run this
def min_subarray_len(target, nums):
    left, total, best = 0, 0, float("inf")
    for right, x in enumerate(nums):
        total += x
        while total >= target:               # valid: record, then shrink to look for a shorter one
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best


def min_subarray_len_brute(target, nums):
    lengths = [j - i for i in range(len(nums)) for j in range(i + 1, len(nums) + 1) if sum(nums[i:j]) >= target]
    return min(lengths, default=0)


for (target, nums), expected in [((7, [2, 3, 1, 2, 4, 3]), 2), ((4, [1, 4, 4]), 1), ((11, [1, 1, 1, 1, 1, 1, 1, 1]), 0), ((3, []), 0)]:
    assert min_subarray_len(target, nums) == expected, (target, nums)
for _ in range(300):
    nums, target = [rng.randint(1, 6) for _ in range(rng.randint(0, 12))], rng.randint(1, 20)
    assert min_subarray_len(target, nums) == min_subarray_len_brute(target, nums)
print("min_subarray_len: all cases + 300 random inputs pass")

## Say it in an interview

- **30-second answer (hashing):** "A hash map trades O(n) memory for O(1) average lookups, so a single pass can remember every value, index or count I have seen — that is what turns a nested-loop pair search into O(n). I pick the key carefully: the value itself, its complement, or a canonical form such as sorted letters."
- **30-second answer (window):** "For contiguous ranges I slide a window: add the element entering on the right, remove the one leaving on the left, and keep the state (sum, counts) up to date incrementally. Each element enters and leaves once, so it is O(n). It needs the rule to be monotone; with negative numbers I switch to prefix sums plus a hash map of prefix counts."
- **Key facts:** dict/set ops are O(1) average, O(n) worst case (collisions); hashing a string costs O(length) the first time; keys must be immutable (tuple, not list); Python randomises string hashes per process, so never rely on set order.
- **Follow-ups:** "Space?" — the hash map is O(n), the window's counter is O(alphabet). "Can you avoid the extra space?" — only by sorting (O(n log n), loses indices). "Stream of data?" — windows and prefix-sum maps work in one pass.
- **Traps:** checking the complement after inserting (pairs an element with itself); letting `left` move backwards (`"abba"`); forgetting `seen = {0: 1}`; using a window when numbers can be negative; comparing whole Counters every step when an O(1) `missing` count would do.

## Next

- [04 · Stacks and queues](04_stacks_and_queues.ipynb) — the monotonic deque gives the *maximum* of every window in O(n)
- [02 · Arrays and two pointers](02_arrays_and_two_pointers.ipynb) — prefix sums and the sorted-array version of Two Sum
- [14 · Tries, union-find and LRU cache](14_tries_union_find_lru_cache.ipynb) — a hash map plus a linked list gives an O(1) LRU cache
- GenAI link: [Caching: exact and semantic](../16_production/03_caching_exact_and_semantic.ipynb) — an exact-match LLM cache is a hash map on a normalised prompt
- Theory: [DSA basics course](../../dsa_basics/index.html) (chapter 6, "Hash tables") · [interview bank](../../ai_interview_prep/index.html)